In [ ]:
import numpy as np
import tensorflow as tf
from PIL import Image

def center_digit(pixels):
    ys, xs = np.where(pixels > 0.15)
    if not len(xs):
        return np.zeros((28, 28), dtype=np.float32)
    digit = pixels[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    scale = 20 / max(digit.shape)
    size = tuple(max(1, round(side * scale)) for side in digit.shape[::-1])
    digit = np.asarray(Image.fromarray((digit * 255).astype(np.uint8)).resize(size, Image.Resampling.LANCZOS)) / 255.0
    canvas = np.zeros((28, 28), dtype=np.float32)
    top, left = (28 - digit.shape[0]) // 2, (28 - digit.shape[1]) // 2
    canvas[top:top + digit.shape[0], left:left + digit.shape[1]] = digit
    return canvas

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = np.stack([center_digit(image.astype("float32") / 255.0) for image in x_train])
x_test = np.stack([center_digit(image.astype("float32") / 255.0) for image in x_test])
print(x_train.shape, x_test.shape)


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Reshape((28, 28, 1)),
    tf.keras.layers.RandomRotation(0.04, fill_mode="constant", fill_value=0),
    tf.keras.layers.RandomTranslation(0.05, 0.05, fill_mode="constant", fill_value=0),
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()


In [ ]:
callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)]
model.fit(x_train, y_train, validation_split=0.1, epochs=15, batch_size=128, callbacks=callbacks)
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"MNIST test accuracy: {accuracy:.2%}")
if accuracy < 0.98:
    raise RuntimeError("La precisión quedó por debajo del 98%; no se guardó un modelo de baja calidad.")
model.save("modelo_mnist.keras")
